In [ ]:
# Importing dependancy libraries
import os
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
import math as m
from collections import Counter
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
stop_list = set(stopwords.words('english'))
# sorted(stop_list)

In [ ]:
# Declaring variables for file path
in_path = r'C:\Users\Natha\Downloads\cran'
doc_source = os.path.join(in_path, 'cran.all.1400.txt')
out_path = os.path.join(in_path, 'preprocessed_cranfieldDocs')

# Declaring variables for query files
query = os.path.join(in_path, 'cran.qry.txt')
preproc_query = os.path.join(in_path, 'preprocessed_queries.txt')

# Declaring variable for file with query relevance values
relevance = os.path.join(in_path, 'cranqrel.txt')

# Checking if the preprocessed docs folder exists already
if not os.path.isdir(out_path):
    os.mkdir(out_path)

# Initiallizing Porter Stemmer object
st = PorterStemmer()

# Initializing regex to remove words with one or two characters length
stop_list = set(stopwords.words('english'))   # faster
shortword = re.compile(r'\W*\b\w{1,2}\b')        # from provided code

In [ ]:
def tokenize(data):
    """Preprocesses the string given as input. Converts to lower case,
    removes the punctuations and numbers, splits on whitespaces, 
    removes stopwords, performs stemming & removes words with one or 
    two characters length.

    Arguments:
        data {string} -- string to be tokenized

    Returns:
        string -- string of tokens generated
    """

    # converting to lower case
    lines = data.lower()

    # removing punctuations by using regular expression
    lines = re.sub('[^A-Za-z]+', ' ', lines)

    # splitting on whitespaces to generate tokens
    tokens = lines.split()

    # removing stop words from the tokens
    clean_tokens = [word for word in tokens if word not in stop_list]

    # stemming the tokens
    stem_tokens = [st.stem(word) for word in clean_tokens]

    # checking for stopwords again
    clean_stem_tokens = [word for word in stem_tokens if word not in stop_list]

    # converting list of tokens to string
    clean_stem_tokens = ' '.join(map(str,  clean_stem_tokens))

    # removing tokens with one or two characters length
    clean_stem_tokens = shortword.sub('', clean_stem_tokens)

    return clean_stem_tokens

In [ ]:
def extractTokens(beautSoup, tag):
    """Extract tokens of the text between a specific SGML <tag>. The function
    calls tokenize() function to generate tokens from the text.

    Arguments:
        beautSoup {bs4.BeautifulSoup} -- soup bs object formed using text of a file
        tag {string} -- target SGML <tag>

    Returns:
        string -- string of tokens extracted from text between the target SGML <tag>
    """

    # extract text of a particular SGML <tag>
    textData = beautSoup.findAll(tag)

    # converting to string
    textData = ''.join(map(str, textData))
    # remove the SGML <tag> from text
    textData = textData.replace(tag, '')

    # calling function to generate tokens from text
    textData = tokenize(textData)

    return textData

In [ ]:
def parse_cranfield_docs(filepath):
    docs = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        current_id = None
        current_section = None
        sections = {'T': '', 'W': ''}

        for line in f:
            line = line.rstrip('\n')
            if line.startswith('.I'):
                if current_id is not None:
                    docs.append({
                        'id': current_id,
                        'title': sections['T'].strip(),
                        'text': sections['W'].strip()
                    })
                current_id = line.split()[1]
                current_section = None
                sections = {'T': '', 'W': ''}
            elif line.startswith('.'): 
                tag = line[1:2]
                current_section = tag if tag in ('T', 'W') else None
            elif current_section in sections:
                text = line.strip()
                if text:
                    sections[current_section] += (' ' if sections[current_section] else '') + text

        if current_id is not None:
            docs.append({
                'id': current_id,
                'title': sections['T'].strip(),
                'text': sections['W'].strip()
            })

    return docs


def parse_cranfield_queries(filepath):
    queries = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        current_id = None
        current_section = None
        current_text = ''

        for line in f:
            line = line.rstrip('\n')
            if line.startswith('.I'):
                if current_id is not None:
                    queries.append({'id': current_id, 'text': current_text.strip()})
                current_id = line.split()[1]
                current_section = None
                current_text = ''
            elif line.startswith('.'): 
                current_section = 'W' if line.startswith('.W') else None
            elif current_section == 'W':
                text = line.strip()
                if text:
                    current_text += (' ' if current_text else '') + text

        if current_id is not None:
            queries.append({'id': current_id, 'text': current_text.strip()})

    return queries

In [ ]:
# Preprocessing all the documents in the Cranfield dataset

"""This cell might take about 20 seconds to run."""
docs = parse_cranfield_docs(doc_source)
all_docs = []

for doc in docs:
    preprocessed_text = tokenize(doc['title'] + ' ' + doc['text'])
    all_docs.append(preprocessed_text)

    outfilepath = os.path.join(out_path, f"doc_{doc['id']}.txt")
    with open(outfilepath, 'w', encoding='utf-8') as outfile:
        outfile.write(preprocessed_text)

In [ ]:
# Preprocessing the queries.txt file
queries_data = parse_cranfield_queries(query)
queries = []

with open(preproc_query, 'w', encoding='utf-8') as new_q:
    for i, entry in enumerate(queries_data):
        query_tokens = tokenize(entry['text'])
        queries.append(query_tokens)
        if i != len(queries_data) - 1:
            new_q.write(query_tokens + '\n')

        else:            new_q.write(query_tokens)

In [ ]:
# The corpus is already loaded into all_docs from the Cranfield parser above.
# The preprocessed files in out_path were written for inspection only.


In [ ]:
# total number of documents is 1400
no_of_docs = len(all_docs)
print(no_of_docs)

In [ ]:
# create a dictionary of key-value pairs where tokens are keys and their occurence in the corpus the value
DF = {}

for i in range(no_of_docs):
    tokens = all_docs[i].split()
    for w in set(tokens):
            # to handle when a new token is encountered
        DF.setdefault(w, set()).add(i)
for i in DF:
    # convert to number of occurences of the token from list of documents where token occurs
    DF[i] = len(DF[i])

In [ ]:
print(DF)

In [ ]:
# count number of unique words in the corpus
vocab_size = len(DF)
print(vocab_size)

In [ ]:
# create vocabulary list of all unique words
vocab = [term for term in DF]
print(vocab)

In [ ]:
doc = 0

# creating dictionary to store tf-idf values for each term in the vocabulary
tf_idf = {}

for i in range(no_of_docs):
    
    tokens = all_docs[i].split()
    
    # counter object to efficiently count number of occurence of a term in a particular document
    counter = Counter(tokens)
    words_count = len(tokens)
    
    for token in np.unique(tokens):
        
        # counting occurence of term in object using counter object
        tf = counter[token]/words_count
        # retrieving df values from DF dictionary
        df = DF.get(token, 0)
        
        # adding 1 to numerator & denominator to avoid divide by 0 error
        idf = np.log((no_of_docs+1)/(df+1))
        
        tf_idf[doc, token] = tf*idf

    doc += 1

In [ ]:
# print(tf_idf)
print(tf_idf)

In [ ]:
# initializing empty vector of vocabulary size
D = np.zeros((no_of_docs, vocab_size))

# creating vector of tf-idf values
vocab_index = {term: i for i, term in enumerate(vocab)}   

for key in tf_idf:
    ind = vocab_index[key[1]]
    D[key[0]][ind] = tf_idf[key]

In [ ]:
# len(D)
print(D)

In [ ]:
def gen_vector(tokens):
    """To create a vector (with repsect to the vocabulary) of the tokens passed as input
    
    Arguments:
        tokens {list} -- list of tokens to be converted
    
    Returns:
        numpy.ndarray -- vector of tokens
    """
    Q = np.zeros((len(vocab)))
    
    counter = Counter(tokens)
    words_count = len(tokens)

    query_weights = {}
    
    for token in np.unique(tokens):
        
        tf = counter[token]/words_count
        df = DF.get(token, 0)
        idf = m.log((no_of_docs+1)/(df+1))

        try:
            ind = vocab_index[token]
            Q[ind] = tf*idf
        except:
            pass 
    return Q

In [ ]:
def cosine_sim(x, y):
    """To calculate cosine similarity between 2 vectors.
    
    Arguments:
        x {numpy.ndarray} -- vector 1
        y {numpy.ndarray} -- vector 2
    
    Returns:
        numpy.float64 -- cosine similarity between vector 1 & vector 2
    """
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    if denom == 0:
        return 0.0
    return np.dot(x, y) / denom

In [ ]:
# Alternative cosine_sim from provided code (without zero check)
def cosine_sim_alt(x, y):
    cos_sim = np.dot(x, y)/(np.linalg.norm(x)*np.linalg.norm(y))
    return cos_sim

In [ ]:
def cosine_similarity(k, query):
    """To determine a ranked list of top k documents in descending order of their
    cosine similarity with the query
    
    Arguments:
        k {integer} -- top k documents to retrieve from 
        query {string} -- query whose cosine similarity is to be computed with the corpus
    
    Returns:
        numpy.ndarray -- list of top k cosine similarities between query and corpus of documents
    """

    tokens = query.split()
      
    d_cosines = []
    
    # vectorize the input query tokens
    query_vector = gen_vector(tokens)
    
    for d in D:
        d_cosines.append(cosine_sim(query_vector, d))
        
    if k == 0:
        # k=0 to retrieve all documents in descending order
        out = np.array(d_cosines).argsort()[::-1]
        
    else:
        # to retrieve the top k documents in descending order    
        out = np.array(d_cosines).argsort()[-k:][::-1]
    
    return out

In [ ]:
# list of all queries from preprocessed queries file
with open(preproc_query, 'r') as query_file:
    queries = query_file.readlines()
print(queries)

In [ ]:
def list_of_docs(k):
    """To generate a ranked list of k top documents in descending order of their cosine similarity 
    calculated against the queries. Output is a list of (query id, document id) pairs.
    
    If k=0 is given as input then list of all documnets in descending order is returned.
    
    Arguments:
        k {integer} -- number of top documents to be retrieved
    
    Returns:
        list -- list of documents in descending order of their cosine similarity
    """
    cos_sims = []
    for i in range(len(queries)):
        cs = [i+1, cosine_similarity(k, queries[i])]
        cos_sims.append(cs)
        
    return cos_sims


In [ ]:
def load_cran_qrels(file_path):
    """
    Reads cranqrel.txt file with 3 columns:
    query_id, doc_id, relevance_score

    Ignores entries with relevance = -1
    """
    qrels = {}

    with open(file_path, 'r') as f:
        for line in f:
            parts = line.split()
            
            if len(parts) != 3:
                continue

            qid = int(parts[0])
            docid = int(parts[1])
            rel = int(parts[2])

            if rel == -1:
                continue

            # keep only strong relevance (1 and 2)
            if rel <= 2:
                qrels.setdefault(qid, []).append(docid)  # 1-based

    return qrels

In [ ]:
# Alternative way to load relevance using pandas (adapted for 3-column format)

colnames=['query', 'docid', 'rel_score'] 
rel = pd.read_csv(relevance, delim_whitespace=True, names=colnames, header=None)
rel.head(10)

# Filter relevant (rel_score > 0)
rel = rel[rel['rel_score'] > 0]

# list of relevant document numbers for a document
rel_list = []

# list of list of relevant document numbers for all documents
query_rel_alt = []
for i in range(1, len(queries)+1):
    rel_list = rel[rel['query']==i]['docid'].to_list()  # 1-based
    
    # append list rel_list to list of list query_rel_alt
    query_rel_alt.append(rel_list)
print(query_rel_alt)

In [ ]:
#to get list of all documents
no_of_top=0
list_of_docs(no_of_top)

In [ ]:
# Load new cranfield qrels dataset

qrels = {}

with open(relevance, "r") as f:
    for line in f:
        parts = line.split()
        
        if len(parts) != 3:
            continue
        
        qid = int(parts[0])
        docid = int(parts[1])
        rel_score = int(parts[2])

        if rel_score == -1:
            continue

        if rel_score <= 3:   # relevant
            if qid not in qrels:
                qrels[qid] = []
            qrels[qid].append(docid - 1)

# Convert into same format your code expects
query_rel = []

for i in range(1, len(queries)+1):
    query_rel.append(qrels.get(i, []))

In [ ]:
# Build graded relevance dictionary
graded_qrels = {}

with open(relevance, "r") as f:
    for line in f:
        parts = line.split()
        
        if len(parts) != 3:
            continue
        
        qid = int(parts[0])
        docid = int(parts[1])
        rel_score = int(parts[2])

        if rel_score == -1:
            continue

        if qid not in graded_qrels:
            graded_qrels[qid] = {}
        
        graded_qrels[qid][docid - 1] = rel_score

In [ ]:
def intersection(lst1, lst2): 
    """To count number of common items between 2 lists
    
    Arguments:
        lst1 {list} -- list 1
        lst2 {list} -- list 2
    
    Returns:
        integer -- number of common items between list 1 & list 2 
    """
    lst3 = [value for value in lst1 if value in lst2] 
    return len(lst3) 

In [ ]:
top = [10, 50, 100, 500]

# for top 100 docs
no_of_top = top[2]
print(no_of_top)

In [ ]:
def calculate_recall_from_results(results, k):
    """To generate list of recall values for each query for given value of k
    
    Arguments:
        results {list} -- list of top documents for each query
        k {integer} -- number of top documents to be retrieved
    
    Returns:
        list -- list of recall values for each query
    """
    recall = []
    for i in range(len(queries)):
        
        retrieved = results[i][1][:k].tolist()
        a = intersection(retrieved, query_rel[i])
        
        # Total number of relevant documents
        b = len(query_rel[i])
        r = a / b if b != 0 else 0
        recall.append(r)
    return recall

# for top 100 docs
calculate_recall_from_results(list_of_docs(100), 100)


In [ ]:
# Alternative calculate_recall from provided code
def calculate_recall_alt(k):
    recall = []
    for i in range(len(queries)):
        a = intersection(list_of_docs(k)[i][1].tolist(), query_rel[i])
        b = len(query_rel[i])
        r = a / b if b != 0 else 0
        recall.append(r)
    return recall

In [ ]:
np.mean(calculate_recall_from_results(list_of_docs(100), 100))

In [ ]:
def calculate_precision_from_results(results, k):
    """To generate list of precision values for each query for given value of k
    
    Arguments:
        results {list} -- list of top documents for each query
        k {integer} -- number of top documents to be retrieved
    
    Returns:
        list -- list of precision values for each query
    """
    precision = []
    for i in range(len(queries)):
        retrieved = results[i][1][:k].tolist()
        a = intersection(retrieved, query_rel[i])
        
        # Total number of documents retrieved
        b = k
        p = a / b if b != 0 else 0
        precision.append(p)
    return precision

# for top 100 docs
calculate_precision_from_results(list_of_docs(100), 100)


In [ ]:
# Alternative calculate_precision from provided code
def calculate_precision_alt(k):
    precision = []
    for i in range(len(queries)):
        a = intersection(list_of_docs(k)[i][1].tolist(), query_rel[i])
        b = k
        p = a / b if b != 0 else 0
        precision.append(p)
    return precision

In [ ]:
def calculate_fscore(k):
    """To generate list of F1-score values for each query for given value of k

    Arguments:
        k {integer} -- number of top documents to be retrieved

    Returns:
        list -- list of F1-score values for each query
    """

    precision_vals = calculate_precision_alt(k)
    recall_vals = calculate_recall_alt(k)

    fscore = []

    for i in range(len(queries)):

        p = precision_vals[i]
        r = recall_vals[i]

        # Avoid division by zero
        if (p + r) == 0:
            f1 = 0.0
        else:
            f1 = (2 * p * r) / (p + r)

        fscore.append(f1)

    return fscore

In [ ]:
def average_precision(ranked_docs, relevant_docs):
    score = 0.0
    hits = 0
    
    for i, doc_id in enumerate(ranked_docs):
        if doc_id in relevant_docs:
            hits += 1
            score += hits / (i + 1)
    
    if len(relevant_docs) == 0:
        return 0.0
    
    return score / len(relevant_docs)

In [ ]:
def mean_average_precision(k):
    results = list_of_docs(k)
    APs = []
    
    for i in range(len(queries)):
        retrieved = results[i][1].tolist()
        relevant = query_rel[i]
        
        ap = average_precision(retrieved, relevant)
        APs.append(ap)
    
    return np.mean(APs)

In [ ]:
print("MAP@100:", mean_average_precision(100))

In [ ]:

np.mean(calculate_precision_from_results(list_of_docs(100), 100))

In [ ]:
# To print the output of answers to questions in Task 2
# Output saved to output.txt file

"""This cell might take about 40 seconds to run."""

from model.VSM import calculate_precision


with open('output.txt', 'w') as f:
    for t in top:
        f.write(f"Top {t} documents in the rank list\n")
        results = list_of_docs(t)
        p = calculate_precision_from_results(results, t)
        r = calculate_recall_from_results(results, t)
        for i in range(len(queries)):
            f.write(f"Query: {i+1} \t Pr: {p[i]} \t Re: {r[i]}\n")
        f.write(f"Avg Precision: {np.mean(p)}\n")
        f.write(f"Avg Recall: {np.mean(r)}\n\n")

for t in top:

    p = calculate_precision_alt(t)
    r = calculate_recall_alt(t)
    f = calculate_fscore(t)

    print("Top {0} documents in the rank list".format(t))

    for i in range(len(queries)):
        print(
            "Query: {0} \t Pr: {1} \t Re: {2} \t F1: {3}".format(
                i + 1,
                round(p[i], 4),
                round(r[i], 4),
                round(f[i], 4)
            )
        )

    print("Avg Precision: {0}".format(round(np.mean(p), 4)))
    print("Avg Recall: {0}".format(round(np.mean(r), 4)))
    print("Avg F1-Score: {0}\n".format(round(np.mean(f), 4)))